In [ ]:
import requests
import json
import io

In [ ]:
# Get the bounding box of the WMNF
baseurl = 'https://nominatim.openstreetmap.org/search'
params = {'q': 'White Mountain National Forest',
          'format': 'json',
          'polygon_geojson': 1}
user_agent = {'User-agent': 'WMNF Pole of Inaccessability Notebook'}

nom_result = requests.get(baseurl, headers=user_agent, params=params)

json_place = json.load(io.BytesIO(nom_result.content))
lat0, lat1, lon0, lon1 = json_place[0]["boundingbox"]

Now that we have a bounding box for the place name "White Mountain National Forest", issue a request for the "highways" that are in that bounding box. In OSM land, "highways" includes paths and trails. I am rather making the assumption that the trails in the WMNF are included in the OSM data, but rendering it will certainly answer that for me in short order. 

In [ ]:
# Issuing a request is a post with some parameters
overpass = f"""
[out:json][timeout:180];
way ["highway"] ({lat0},{lon0},{lat1},{lon1});
(._;>;);
out;
"""
result = requests.post('http://overpass-api.de/api/interpreter', data=overpass, headers=user_agent)

# Write it to a file. Note, this didn't check if the output was any good. 
json_wmnf_roads = json.load(io.BytesIO(result.content))
with open("wmnf_roads.json", "w") as outfile:
    json.dump(json_wmnf_roads, outfile, indent=4)

Now get the actual polygons that define the boundaries of the WMNF. This is a complicated thing, because there are multiple areas that are in the WMNF, and those areas have inholdings that are _not_ in the WMNF, so you can be within a larger polygon but in an inholding and so not "in" the WMNF. 

In [ ]:
# Now actually get the boundaries of the WMNF and dump that to a file
nlookup_url = 'https://nominatim.openstreetmap.org/lookup'
params = {'osm_ids':f'R{json_place[0]["osm_id"]}',
          'format':'json',
          'polygon_geojson':1}

bound_result = requests.get(nlookup_url, headers=user_agent, params=params)
bound_json = json.load(io.BytesIO(bound_result.content))
with open("wmnf_boundary.json", "w") as outfile:
    json.dump(bound_json, outfile, indent=4)